### Merge lora with base model

In [1]:
import os
from merge import merge
base_model_id = "Qwen/Qwen2.5-Coder-0.5B"
lora_path = "../loras/qwen25-05-ft"
merged_path = "merged/qwen25-05-ft"
gguf_f16_path = "quantized/qwen25-05-ft.gguf"
# gguf_8bit_path = "quantized/qwen25-05i-base-Q8_0.gguf"
gguf_4bit_path = "quantized/qwen25-05-ft-Q4KM.gguf"
# gguf_2bit_path = "quantized/qwen25-05i-base-Q2_K.gguf"
llama_cpp_path = "llama.cpp"
llama_distribution_path = "llama-b7475-bin-win-cpu-x64"

c:\Users\Anh\.conda\envs\ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.7.0+cu128 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0331 08:35:00.582000 24676 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
merge(base_model_id, lora_path, merged_path)

Model merged completed, saved to merged/qwen25-05-ft


### Quantize model

Convert to gguf format

In [3]:
import subprocess
args = [
    "python", os.path.join(llama_cpp_path, "convert_hf_to_gguf.py"),
    merged_path,
    "--outfile", gguf_f16_path
]
result = subprocess.run(
    args,
    capture_output=True,
    text=True
)
print(result.stdout)
print(result.stderr)


INFO:hf-to-gguf:Loading model: qwen25-05-ft
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.bfloat16 --> F16, shape = {896, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.bfloat16 --> F32, shape = {896}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.bfloat16 --> F16, shape = {4864, 896}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.bfloat16 --> F16, shape = {896, 4864}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.bfloat16 --> F16, shape = {896, 4864}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.bfloat16 --> F32, shape = {896}
INFO:hf-to-gguf:blk.0.attn_k.bias,         torch.bfloat16 --> F32, shape = {128}
INFO:hf-to-gguf:blk.0.attn_k.weight,       torch.bfloat16 --> F16, shape = {896, 128}
INFO:hf-to-gguf:blk.0.attn_output.weight,  torch.bfloat16

Quantize to 4bit

In [4]:
import subprocess

args = [
    os.path.join(llama_distribution_path, "llama-quantize.exe"),
    gguf_f16_path,
    gguf_4bit_path,
    "Q4_K_M",
    "8"
]
result = subprocess.run(
    args,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)


main: quantize time =  3122.39 ms
main:    total time =  3122.39 ms

main: build = 7475 (8ea958d4d)
main: built with Clang 19.1.5 for Windows x86_64
main: quantizing 'quantized/qwen25-05-ft.gguf' to 'quantized/qwen25-05-ft-Q4KM.gguf' as Q4_K_M using 8 threads
llama_model_loader: loaded meta data with 24 key-value pairs and 290 tensors from quantized/qwen25-05-ft.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwen25 05 Ft
llama_model_loader: - kv   3:                         general.size_label str              = 494M
llama_model_loader: - kv   4:                          qwen2.block_count u32              = 24
llama_model

Quantize to 8bit

In [ ]:
import subprocess

args = [
    os.path.join(llama_distribution_path, "llama-quantize.exe"),
    gguf_f16_path,
    gguf_8bit_path,
    "Q8_0",
    "8"
]
result = subprocess.run(
    args,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

Quantize to 2bit

In [ ]:
import subprocess

args = [
    os.path.join(llama_distribution_path, "llama-quantize.exe"),
    gguf_f16_path,
    gguf_4bit_path,
    "Q2_K",
    "8"
]
result = subprocess.run(
    args,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)